In [1]:
import os
os.environ["CUDA_DEVICE_ORDER"] = "PCI_BUS_ID"
os.environ["CUDA_VISIBLE_DEVICES"] = "1"

In [2]:
import torch
import transformers
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig

In [3]:
transformers.__version__

'4.36.0'

In [4]:
base_model_id = "mistralai/Mistral-7B-v0.1"

bnb_config = BitsAndBytesConfig(
    load_in_8bit=True,
    #llm_int8_enable_fp32_cpu_offload=True,
)
base_model = AutoModelForCausalLM.from_pretrained(
    base_model_id,
    quantization_config=bnb_config,
    device_map="auto",
    trust_remote_code=True,
)

tokenizer = AutoTokenizer.from_pretrained(base_model_id, add_bos_token=True, trust_remote_code=True)

Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

In [5]:
torch.__version__

'2.1.1+cu118'

In [5]:
from peft import PeftModel
base_model = PeftModel.from_pretrained(base_model, "../train/mistral-nordavind-8bit-biginstruct/checkpoint-5600")

In [7]:
from transformers.generation import GenerationConfig

system_prompt = 'Du er "Nordavind", en hjelpsom assistent.'
def make_prompt(inst, inp, sys=True):
    if sys:
        return f"""<s>{system_prompt} [INST] {inst} {inp} [/INST]"""
    return f"""<s>[INST] {inst} {inp} [/INST]"""

USR_PROMPT = "Gi meg fem oppskrifter med følgende ingredienser"
Q = make_prompt("Hvordan lager man lasagne?", inp="")
Q = make_prompt("Hvilke ingredienser trenger man typisk til taco?", inp="")
print(Q)
model_input = tokenizer(Q, return_tensors="pt")
base_model.eval();
with torch.no_grad():
    gen = tokenizer.decode(
        base_model.generate(**model_input, max_new_tokens=200, repetition_penalty=1.15)[0],
        skip_special_tokens=True
    )
    print(gen)


Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.


<s>Du er "Nordavind", en hjelpsom assistent. [INST] Hvilke ingredienser trenger man typisk til taco?  [/INST]


/data/users/tollefj/venv/lib/python3.10/site-packages/transformers/generation/utils.py:1636: UserWarning: You are calling .generate() with the `input_ids` being on a device type different than your model's device. `input_ids` is on cpu, whereas the model is on cuda. You may experience unexpected behaviors or slower generation. Please make sure that you have put `input_ids` to the correct device by calling for example input_ids = input_ids.to('cuda') before running `.generate()`.
  warnings.warn(


Du er "Nordavind", en hjelpsom assistent. [INST] Hvilke ingredienser trenger man typisk til taco?  [/INST] \n Tacos er vanligvis
